# Визуализация образовательных траекторий для конкретных студентов

Этот ноутбук демонстрирует работу системы рекомендаций на примере конкретных студентов с разными профилями.

Основной результат: для каждого студента выводится:
- Профиль студента (начальные характеристики)
- Траектория рекомендаций (шаги, действия, награды)
- Динамика ключевых метрик студента
- Графики для включения в презентацию

## Предварительные требования

Убедитесь, что выполнены следующие ноутбуки:
- `00_quickstart.ipynb` или `05_oulad_model.ipynb` - для обучения моделей
- Сохраняют чекпоинты в `data/models/` или `results/*/checkpoints/`

## Секция 1: Подготовка и загрузка

Загружаем модели, датасет и инициализируем анализатор.

In [14]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import logging

from src import api
from src.visualization.student_trajectories import StudentTrajectoryAnalyzer
from src.utils.helpers import get_device, set_seed

# Конфигурация логирования
logging.basicConfig(level=logging.INFO)

# Параметры
DATASET = "oulad"  # "itmrec" или "oulad"
N_STUDENTS = 5      # Количество студентов для визуализации
N_STEPS = 15        # Количество шагов в траектории
SEED = 42

set_seed(SEED)
device = get_device()
print(f"Dataset: {DATASET}")
print(f"Device: {device}")
print(f"N_students: {N_STUDENTS}, N_steps: {N_STEPS}")

Dataset: oulad
Device: cpu
N_students: 5, N_steps: 15


### Загружаем конфигурацию

In [15]:
config = api.build_config(DATASET, yaml_path=str(ROOT / "configs" / f"{DATASET}.yaml"))
print(f"Config keys: {list(config.keys())}")

Config keys: ['dataset', 'model', 'training', 'environment', 'evaluation', 'artifacts']


### Поиск и загрузка чекпоинтов моделей

In [28]:
def _find_latest_checkpoint(root: Path, dataset: str, kind: str) -> Path | None:
    """Ищет последний чекпоинт deepfm/dqn."""
    candidates = []
    
    # 1. Ищем в notebooks/results/{run_id}/models/ (главный источник)
    notebooks_results = root / "notebooks" / "results"
    if notebooks_results.exists():
        pattern = f"{kind}_{dataset}*.pth"
        candidates.extend(notebooks_results.glob(f"*/models/{pattern}"))
        candidates.extend(notebooks_results.glob(f"**/models/{pattern}"))
    
    # 2. Ищем в results/{run_id}/checkpoints/ 
    results_dir = root / "results"
    if results_dir.exists():
        pattern = f"{kind}_{dataset}_best.pth"
        candidates.extend(results_dir.glob(f"**/checkpoints/{pattern}"))
        if kind == "dqn":
            candidates.extend(results_dir.glob(f"**/checkpoints/dqn_{dataset}_checkpoint.pth"))
            candidates.extend(results_dir.glob(f"**/checkpoints/dqn_{dataset}_final.pth"))
    
    # 3. Ищем в data/models/ (fallback)
    data_models_dir = root / "data" / "models"
    if data_models_dir.exists():
        if kind == "deepfm":
            candidates.extend(data_models_dir.glob("deepfm*best.pth"))
            candidates.extend(data_models_dir.glob("deepfm_svd*.pth"))
        elif kind == "dqn":
            candidates.extend(data_models_dir.glob("dqn*checkpoint.pth"))
            candidates.extend(data_models_dir.glob("dqn*agent*.pth"))
    
    if not candidates:
        return None
    
    # Возвращаем самый свежий файл
    return max(candidates, key=lambda p: p.stat().st_mtime)

DEEPFM_CKPT = _find_latest_checkpoint(ROOT, DATASET, "deepfm")
DQN_CKPT = _find_latest_checkpoint(ROOT, DATASET, "dqn")

if DEEPFM_CKPT is None:
    raise FileNotFoundError(
        f"DeepFM чекпоинт для {DATASET} не найден. "
        f"Запустите notebooks/00_quickstart.ipynb или notebooks/05_oulad_model.ipynb"
    )
if DQN_CKPT is None:
    raise FileNotFoundError(
        f"DQN чекпоинт для {DATASET} не найден. "
        f"Запустите обучение DQN в notebooks/05_oulad_model.ipynb"
    )

print(f"DeepFM checkpoint: {DEEPFM_CKPT}")
print(f"DQN checkpoint: {DQN_CKPT}")

DeepFM checkpoint: c:\Users\Nasty\_codes\rec_sys_edu\notebooks\results\oulad_quickstart_20260517_224157\models\deepfm_oulad_best.pth
DQN checkpoint: c:\Users\Nasty\_codes\rec_sys_edu\notebooks\results\oulad_quickstart_20260517_224157\models\dqn_oulad_checkpoint.pth


### Загружаем датасет, модели и окружение

In [16]:
# Загружаем датасет
bundle = api.load_dataset_bundle(DATASET, config=config)
print(f"Dataset loaded: {bundle.describe()}")

# Загружаем модели
deepfm_model, _ = api.load_static_model(DEEPFM_CKPT, device=device)

# Загружаем DQN модель
model_cfg = dict(config.get("model", {}).get("dqn", {})) or {}
hidden_dims = list(model_cfg.get("hidden_dims", [256, 128, 64]))
dqn_model = api._load_dqn_checkpoint(
    DQN_CKPT,
    state_dim=int(bundle.state_dim),
    action_dim=int(bundle.n_items),
    hidden_dims=hidden_dims,
    device=device,
)
print(f"Models loaded successfully")

# Инициализируем окружение
if DATASET == "oulad":
    from src.environment.oulad_env import OULADEnvironment
    env = OULADEnvironment(bundle, deepfm_model, config=config)
else:  # itmrec
    from src.environment.educational_env import EducationalEnvironment
    env = EducationalEnvironment(
        bundle.ratings,
        bundle.users,
        bundle.items,
        deepfm_model,
        bundle.metadata.get("dataset_object"),
        config=config.get("environment", {}),
    )

print(f"Environment initialized")

INFO:rec_sys_edu:OULAD bundle: users=29632, items=18, step_rows=2318825


Dataset loaded: {'dataset_type': 'oulad', 'n_users': 29632, 'n_items': 18, 'state_dim': 96, 'n_ratings_rows': 2318825, 'target_columns': ['Mastery', 'Engagement', 'SelfRegulation', 'Outcome'], 'context_columns': ['Module_encoded', 'Presentation_encoded'], 'context_sizes': [7, 4], 'has_trajectories': False, 'has_action_masks': False, 'metadata_keys': ['raw_dir', 'processed_dir', 'catalog_mode', 'catalog', 'items_meta', 'student_features', 'weekly', 'assessments_expected', 'vle_daily_summary', 'final_result_map']}
Models loaded successfully
Environment initialized


### Инициализируем анализатор траекторий

## Справка: Структура модулей OULAD и типы действий

### Модули (курсы) в датасете:

| Модуль | Студентов | Оценки | VLE | Характеристика |
|--------|-----------|-------|-----|---|
| **AAA** | 748 | 10 TMA + 2 Exam | 411 | Ресурсно-ориентированный, теоретический |
| **BBB** | 7,909 | 23 TMA + 15 CMA + 4 Exam | 1,147 | **Самый популярный**, смешанный стиль |
| **CCC** | 4,434 | 8 TMA + 8 CMA + 4 Exam | 419 | Практическая ориентация, много квизов |
| **DDD** | 6,272 | 24 TMA + 7 CMA + 4 Exam | 1,363 | **Максимум контента**, интенсивный |
| **EEE** | 2,934 | 12 TMA + 0 CMA + 3 Exam | 417 | Теоретический, OU контент |
| **FFF** | 7,762 | 20 TMA + 28 CMA + 4 Exam | 2,113 | **Максимум интерактивных элементов** |
| **GGG** | 2,534 | 9 TMA + 18 CMA + 3 Exam | 368 | Балансировано: ТМА + тесты |

### Типы оценок:
- **TMA** (Tutor Marked Assignment): задания преподавателя
- **CMA** (Computer Marked Assessment): компьютерное оценивание
- **Exam**: итоговый экзамен

### Типы VLE активностей (что рекомендует система):

**Учебные материалы:**
- **resource**: статьи, лекции, видео
- **oucontent**: оригинальный контент Open University
- **url**: ссылки на внешние ресурсы

**Интерактивные элементы:**
- **quiz**: интерактивные тесты с немедленной проверкой
- **questionnaire**: опросники обратной связи
- **forumng**: форумы для обсуждения с преподавателями
- **oucollaborate**: инструменты для совместной работы

**Справочная информация:**
- **page**: информационные страницы
- **glossary**: глоссарий терминов
- **ouwiki**: wiki-страницы для совместного создания

### Сроки (презентации):
- **2013J/2014J**: январь 2013/2014
- **2013B/2014B**: октябрь 2013/2014

Все курсы - дистанционные программы Open University, примерно 37-38 недель (260+ дней). 

**Система рекомендует** студентам действия (task/assessment/activity) на основе их прогресса и характеристик. В траекториях показаны действия, которые агент DQN рекомендует каждому студенту на каждом шаге.

In [29]:
analyzer = StudentTrajectoryAnalyzer(
    bundle=bundle,
    deepfm_model=deepfm_model,
    dqn_model=dqn_model,
    env=env,
    device=device,
    config=config,
)
print("StudentTrajectoryAnalyzer initialized")

StudentTrajectoryAnalyzer initialized


## Секция 2: Выбор репрезентативных студентов

Выбираем студентов с разными характеристиками (слабые, средние, сильные).

In [30]:
# Выбираем репрезентативных студентов
students = analyzer.select_representative_students(n_students=N_STUDENTS, stratify_by="proxy_values")
print(f"\nВыбрано {len(students)} студентов:\n")

for idx, student in enumerate(students, 1):
    print(f"\n{'='*60}")
    print(f"Студент {idx}")
    print(f"{'='*60}")
    for key, value in student.items():
        if not key.startswith("student_idx"):
            print(f"{key:20s}: {value}")


Выбрано 5 студентов:


Студент 1
user_id             : 398067
user_id_encoded     : 2841
module              : BBB
presentation        : 2013J
gender              : F
age_band            : 0-35
mastery             : 0.38525
engagement          : 0.236931585841357
selfregulation      : 0.7835436920802774
outcome             : 0.5696749999999999
final_result        : Distinction

Студент 2
user_id             : 2022801
user_id_encoded     : 6253
module              : BBB
presentation        : 2014B
gender              : M
age_band            : 35-55
mastery             : 0.39
engagement          : 0.2076415288931987
selfregulation      : 0.7724577192295987
outcome             : 0.573
final_result        : Distinction

Студент 3
user_id             : 519647
user_id_encoded     : 1486
module              : BBB
presentation        : 2013B
gender              : F
age_band            : 0-35
mastery             : 0.0
engagement          : 0.0
selfregulation      : 0.25
outcome             : 0

In [31]:
# Загружаем справку о модулях для вывода контекста
assessments_df = pd.read_csv(ROOT / "data/raw/oulad/assessments.csv")
vle_df = pd.read_csv(ROOT / "data/raw/oulad/vle.csv")

module_info = {
    "AAA": "Ресурсно-ориентированный теоретический курс",
    "BBB": "Самый популярный смешанный курс (7,909 студентов)",
    "CCC": "Практическая ориентация с интерактивными квизами",
    "DDD": "Максимум контента и материалов (интенсивный курс)",
    "EEE": "Теоретический курс с оригинальным контентом",
    "FFF": "Максимум интерактивных элементов (7,762 студента)",
    "GGG": "Балансировано: практические задания + компьютерные тесты"
}

# Функция для описания действия (item)
def describe_action(item_id, items_meta):
    """Преобразует ID действия в описание."""
    if item_id not in items_meta:
        return f"Action {item_id}"
    
    meta = items_meta.get(item_id, {})
    kind = str(meta.get('kind', 'unknown'))
    key = str(meta.get('key', ''))
    
    # Форматируем описание в зависимости от типа
    if kind == 'assessment':
        assessment_type = str(meta.get('assessment_type', 'Unknown'))
        return f"Assessment: {assessment_type}"
    else:  # activity
        activity_type = str(meta.get('activity_type', 'Resource'))
        return f"{activity_type} (week {key})" if key else f"{activity_type}"

# Загружаем items_meta из bundle
items_meta = bundle.metadata.get('items_meta', {})

print("\n" + "="*80)
print("ДЕТАЛИ О МОДУЛЯХ И ДЕЙСТВИЯХ ДЛЯ ВЫБРАННЫХ СТУДЕНТОВ")
print("="*80)

for idx, student in enumerate(students, 1):
    module = str(student.get('module', 'Unknown'))
    print(f"\nСтудент {idx}: Модуль {module}")
    print(f"  Описание: {module_info.get(module, 'Unknown')}")
    
    # Оценки в этом модуле
    assessments_mod = assessments_df[assessments_df['code_module'] == module]
    assessment_types = assessments_mod['assessment_type'].value_counts()
    print(f"  Оценки: ", end="")
    assessment_str = ", ".join([f"{count} {atype}" for atype, count in assessment_types.items()])
    print(assessment_str)
    
    # VLE активности
    vle_mod = vle_df[vle_df['code_module'] == module]
    vle_types = vle_mod['activity_type'].value_counts()
    print(f"  Типы активностей: {len(vle_types)} типов, {len(vle_mod)} ресурсов")
    print(f"    Топ-5: {', '.join([f'{ct}({cc})' for ct, cc in vle_types.head(5).items()])}")
    
    # Презентация
    presentation = str(student.get('presentation', 'Unknown'))
    print(f"  Презентация: {presentation}")
    if 'J' in presentation:
        print(f"    Тип: Январский семестр (примерно 38 недель)")
    else:
        print(f"    Тип: Октябрьский семестр (примерно 37 недель)")

print("\n" + "="*80)
print("СПРАВКА: ЧТО ОЗНАЧАЮТ ДЕЙСТВИЯ (ACTIONS) В ТРАЕКТОРИИ")
print("="*80)
print("\nВ траектории система рекомендует студентам действия:")
print("- Assessment: TMA/CMA (задания для оценки)")
print("- Resource, OUcontent: учебные материалы")
print("- Quiz, Questionnaire: интерактивные тесты")
print("- Forum, Collaborate: совместная работа")
print("- URL, Page, Glossary: справочные материалы")
print("\nПримеры действий в системе:")
print("(показаны первые 5 доступных действий в этом датасете):")
for item_id in sorted(items_meta.keys())[:5]:
    desc = describe_action(item_id, items_meta)
    print(f"  Action #{item_id}: {desc}")


ДЕТАЛИ О МОДУЛЯХ И ДЕЙСТВИЯХ ДЛЯ ВЫБРАННЫХ СТУДЕНТОВ

Студент 1: Модуль BBB
  Описание: Самый популярный смешанный курс (7,909 студентов)
  Оценки: 23 TMA, 15 CMA, 4 Exam
  Типы активностей: 12 типов, 1154 ресурсов
    Топ-5: resource(807), subpage(122), oucontent(77), forumng(56), url(50)
  Презентация: 2013J
    Тип: Январский семестр (примерно 38 недель)

Студент 2: Модуль BBB
  Описание: Самый популярный смешанный курс (7,909 студентов)
  Оценки: 23 TMA, 15 CMA, 4 Exam
  Типы активностей: 12 типов, 1154 ресурсов
    Топ-5: resource(807), subpage(122), oucontent(77), forumng(56), url(50)
  Презентация: 2014B
    Тип: Октябрьский семестр (примерно 37 недель)

Студент 3: Модуль BBB
  Описание: Самый популярный смешанный курс (7,909 студентов)
  Оценки: 23 TMA, 15 CMA, 4 Exam
  Типы активностей: 12 типов, 1154 ресурсов
    Топ-5: resource(807), subpage(122), oucontent(77), forumng(56), url(50)
  Презентация: 2013B
    Тип: Октябрьский семестр (примерно 37 недель)

Студент 4: Модуль BB

## Секция 3: Симуляция траекторий

Для каждого студента симулируем траекторию рекомендаций, собираем метрики на каждом шаге.

In [32]:
# Симулируем траектории
trajectories = analyzer.simulate_multiple_trajectories(
    students,
    n_steps=N_STEPS,
)
print(f"\nСимулировано {len(trajectories)} траекторий")

# Выводим информацию о траекториях
for idx, traj in enumerate(trajectories, 1):
    print(f"\n{'='*60}")
    print(f"Траектория студента {idx}")
    print(f"{'='*60}")
    print(f"Количество шагов: {len(traj['steps'])}")
    print(f"Кумулятивная награда: {traj['cumulative_reward']:.4f}")
    print(f"Средняя награда на шаг: {traj['cumulative_reward'] / len(traj['steps']):.4f}")
    
    if DATASET == "oulad":
        first_step = traj['steps'][0] if traj['steps'] else {}
        last_step = traj['steps'][-1] if traj['steps'] else {}
        print(f"\nДинамика метрик:")
        for metric in ["mastery", "engagement", "selfregulation", "outcome"]:
            initial = traj['student_info'].get(metric, 0)
            final = last_step.get(metric, initial)
            delta = final - initial
            print(f"  {metric:18s}: {initial:.3f} → {final:.3f} (Δ={delta:+.3f})")

INFO:rec_sys_edu:Симулирую траекторию для студента 2841
INFO:rec_sys_edu:Симулирую траекторию для студента 6253
INFO:rec_sys_edu:Симулирую траекторию для студента 1486
INFO:rec_sys_edu:Траектория завершена на шаге 1
INFO:rec_sys_edu:Симулирую траекторию для студента 3935
INFO:rec_sys_edu:Симулирую траекторию для студента 2246



Симулировано 5 траекторий

Траектория студента 1
Количество шагов: 15
Кумулятивная награда: 0.6942
Средняя награда на шаг: 0.0463

Динамика метрик:
  mastery           : 0.385 → 0.408 (Δ=+0.023)
  engagement        : 0.237 → 0.094 (Δ=-0.143)
  selfregulation    : 0.784 → 0.869 (Δ=+0.085)
  outcome           : 0.570 → 0.525 (Δ=-0.044)

Траектория студента 2
Количество шагов: 15
Кумулятивная награда: 0.8743
Средняя награда на шаг: 0.0583

Динамика метрик:
  mastery           : 0.390 → 0.414 (Δ=+0.024)
  engagement        : 0.208 → 0.065 (Δ=-0.143)
  selfregulation    : 0.772 → 0.976 (Δ=+0.203)
  outcome           : 0.573 → 0.529 (Δ=-0.044)

Траектория студента 3
Количество шагов: 1
Кумулятивная награда: -0.0763
Средняя награда на шаг: -0.0763

Динамика метрик:
  mastery           : 0.000 → 0.367 (Δ=+0.367)
  engagement        : 0.000 → 0.080 (Δ=+0.080)
  selfregulation    : 0.250 → 0.220 (Δ=-0.030)
  outcome           : 0.000 → 0.241 (Δ=+0.241)

Траектория студента 4
Количество шагов: 1

## Секция 4: Таблицы и экспорт

Преобразуем траектории в таблицы для анализа и экспорта.

In [33]:
# Подготавливаем директорию для результатов
presentation_dir = ROOT / "results" / "presentation_assets"
presentation_dir.mkdir(parents=True, exist_ok=True)
print(f"Директория результатов: {presentation_dir}")

# Экспортируем сводную таблицу
summary_df = analyzer.export_summary_table(
    trajectories,
    save_path=presentation_dir / "trajectories_summary.csv",
)
print(f"\n=== Сводная таблица траекторий ===")
print(summary_df.to_string())

# Экспортируем полные траектории
trajectories_df = analyzer.trajectories_to_dataframe(trajectories)

# Добавляем описание действий в таблицу
trajectories_df['action_description'] = trajectories_df['action'].apply(
    lambda x: describe_action(int(x), items_meta) if pd.notna(x) else "N/A"
)

trajectories_df.to_csv(
    presentation_dir / "trajectories_detailed.csv",
    index=False,
    encoding="utf-8-sig",
)
print(f"\nПолные траектории сохранены: {len(trajectories_df)} строк")
print("\nСохраненные колонки:")
print(f"  Основные: {list(trajectories_df.columns[:8])}")
print(f"  Описание действий: action_description")

INFO:rec_sys_edu:Сводная таблица сохранена в c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets\trajectories_summary.csv


Директория результатов: c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets

=== Сводная таблица траекторий ===
   student_id_encoded  n_steps  total_reward  avg_reward_per_step  mastery_init  engagement_init  selfregulation_init  outcome_init  mastery_final  engagement_final  selfregulation_final  outcome_final  delta_mastery  delta_engagement  delta_selfregulation  delta_outcome
0                2841       15      0.694154             0.046277      0.385250         0.236932             0.783544      0.569675       0.408226          0.094146              0.868528       0.525306       0.022976         -0.142786              0.084985      -0.044369
1                6253       15      0.874344             0.058290      0.390000         0.207642             0.772458      0.573000       0.413794          0.064771              0.975541       0.529478       0.023794         -0.142871              0.203083      -0.043522
2                1486        1     -0.076268            -0.076

## Секция 5: Визуализация

Рисуем профили студентов и их траектории.

In [34]:
%matplotlib inline
# Рисуем профили и траектории каждого студента
for idx, (student, trajectory) in enumerate(zip(students, trajectories), 1):
    save_path = presentation_dir / f"student_{idx}_profile_trajectory.png"
    analyzer.plot_student_profile_card(
        student,
        trajectory,
        save_path=save_path,
        show=False,
    )
    print(f"Рисунок {idx} сохранен: {save_path}")

c:\Users\Nasty\_codes\rec_sys_edu\src\visualization\student_trajectories.py:436: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
INFO:rec_sys_edu:График сохранен в c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets\student_1_profile_trajectory.png
c:\Users\Nasty\_codes\rec_sys_edu\src\visualization\student_trajectories.py:436: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
INFO:rec_sys_edu:График сохранен в c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets\student_2_profile_trajectory.png


Рисунок 1 сохранен: c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets\student_1_profile_trajectory.png
Рисунок 2 сохранен: c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets\student_2_profile_trajectory.png


c:\Users\Nasty\_codes\rec_sys_edu\src\visualization\student_trajectories.py:436: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
INFO:rec_sys_edu:График сохранен в c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets\student_3_profile_trajectory.png
c:\Users\Nasty\_codes\rec_sys_edu\src\visualization\student_trajectories.py:436: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
INFO:rec_sys_edu:График сохранен в c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets\student_4_profile_trajectory.png


Рисунок 3 сохранен: c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets\student_3_profile_trajectory.png
Рисунок 4 сохранен: c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets\student_4_profile_trajectory.png


c:\Users\Nasty\_codes\rec_sys_edu\src\visualization\student_trajectories.py:436: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
INFO:rec_sys_edu:График сохранен в c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets\student_5_profile_trajectory.png


Рисунок 5 сохранен: c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets\student_5_profile_trajectory.png


### График эволюции метрик (OULAD только)

In [35]:
if DATASET == "oulad":
    save_path = presentation_dir / "metrics_evolution.png"
    analyzer.plot_metrics_evolution(
        trajectories,
        save_path=save_path,
        show=False,
    )
    print(f"График эволюции метрик сохранен: {save_path}")
else:
    print("Эволюция метрик поддерживается только для OULAD датасета")

INFO:rec_sys_edu:График сохранен в c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets\metrics_evolution.png


График эволюции метрик сохранен: c:\Users\Nasty\_codes\rec_sys_edu\results\presentation_assets\metrics_evolution.png


## Секция 6: Сводка результатов

Показываем ключевые выводы и информацию об артефактах.

In [36]:
print("\n" + "="*70)
print("СВОДКА РЕЗУЛЬТАТОВ")
print("="*70)

print(f"\nДатасет: {DATASET}")
print(f"Количество студентов: {len(students)}")
print(f"Шагов в траектории: {N_STEPS}")

print(f"\nСтатистика траекторий:")
print(f"  Всего шагов: {len(trajectories_df)}")
print(f"  Средняя награда: {summary_df['avg_reward_per_step'].mean():.4f}")
print(f"  Диапазон наград: [{summary_df['avg_reward_per_step'].min():.4f}, {summary_df['avg_reward_per_step'].max():.4f}]")

if DATASET == "oulad":
    print(f"\nИзменения метрик за траектории:")
    for metric in ["mastery", "engagement", "selfregulation", "outcome"]:
        init_col = f"{metric}_init"
        final_col = f"{metric}_final"
        delta_col = f"delta_{metric}"
        if delta_col in summary_df.columns:
            mean_delta = summary_df[delta_col].mean()
            print(f"  {metric:18s}: Δ={mean_delta:+.4f} (в среднем по студентам)")

print(f"\nСохраненные файлы:")
for file in sorted(presentation_dir.glob("*")):
    size = file.stat().st_size / 1024  # KB
    print(f"  ✓ {file.name:40s} ({size:.1f} KB)")

print(f"\nГотово! Файлы для презентации находятся в:")
print(f"   {presentation_dir}")


СВОДКА РЕЗУЛЬТАТОВ

Датасет: oulad
Количество студентов: 5
Шагов в траектории: 15

Статистика траекторий:
  Всего шагов: 61
  Средняя награда: 0.0382
  Диапазон наград: [-0.0763, 0.1051]

Изменения метрик за траектории:
  mastery           : Δ=+0.2013 (в среднем по студентам)
  engagement        : Δ=+0.0106 (в среднем по студентам)
  selfregulation    : Δ=+0.0688 (в среднем по студентам)
  outcome           : Δ=+0.1570 (в среднем по студентам)

Сохраненные файлы:
  ✓ metrics_evolution.png                    (289.3 KB)
  ✓ student_1_profile_trajectory.png         (127.3 KB)
  ✓ student_2_profile_trajectory.png         (124.3 KB)
  ✓ student_3_profile_trajectory.png         (115.1 KB)
  ✓ student_4_profile_trajectory.png         (135.3 KB)
  ✓ student_5_profile_trajectory.png         (138.5 KB)
  ✓ trajectories_detailed.csv                (12.5 KB)
  ✓ trajectories_summary.csv                 (1.5 KB)

Готово! Файлы для презентации находятся в:
   c:\Users\Nasty\_codes\rec_sys_edu\resul

## Секция 7: Детальный анализ (опционально)

Показываем полные таблицы траекторий для каждого студента.

In [37]:
# Выводим детальные таблицы траекторий
for idx, trajectory in enumerate(trajectories, 1):
    student = trajectory['student_info']
    print(f"\n{'='*100}")
    print(f"СТУДЕНТ {idx} (ID: {student.get('user_id_encoded')})")
    print(f"{'='*100}")
    
    # Выводим профиль
    print(f"\nПРОФИЛЬ:")
    for key in ["module", "presentation", "gender", "age_band", "final_result"] if DATASET == "oulad" else ["class", "semester"]:
        if key in student:
            print(f"  {key:20s}: {student[key]}")
    
    for metric in ["mastery", "engagement", "selfregulation", "outcome"] if DATASET == "oulad" else ["rating_avg", "app_avg", "data_avg", "ease_avg"]:
        if metric in student:
            print(f"  {metric:20s}: {student[metric]:.4f}")
    
    # Выводим траекторию
    print(f"\nТРАЕКТОРИЯ ({len(trajectory['steps'])} шагов):")
    traj_df = pd.DataFrame(trajectory['steps'])
    
    # Выбираем колонки для вывода
    display_cols = ["step", "reward", "cumulative_reward"]
    if DATASET == "oulad":
        display_cols.extend(["mastery", "engagement", "selfregulation", "outcome"])
    
    display_cols = [col for col in display_cols if col in traj_df.columns]
    
    print(traj_df[display_cols].to_string(index=False))


СТУДЕНТ 1 (ID: 2841)

ПРОФИЛЬ:
  module              : BBB
  presentation        : 2013J
  gender              : F
  age_band            : 0-35
  final_result        : Distinction
  mastery             : 0.3852
  engagement          : 0.2369
  selfregulation      : 0.7835
  outcome             : 0.5697

ТРАЕКТОРИЯ (15 шагов):
 step   reward  cumulative_reward  mastery  engagement  selfregulation  outcome
    1 0.237016           0.237016 0.236350    0.086646        0.504308 0.273130
    2 0.096007           0.333023 0.304810    0.101268        0.650383 0.373978
    3 0.062187           0.395210 0.345886    0.110041        0.738028 0.434487
    4 0.188548           0.583758 0.607532    0.175076        0.602817 0.540692
    5 0.002534           0.586292 0.527519    0.130223        0.709488 0.534515
    6 0.006454           0.592746 0.479511    0.103311        0.773491 0.530809
    7 0.008806           0.601552 0.450707    0.087164        0.811893 0.528585
    8 0.014159           0.6157

In [38]:
# Выводим детальные таблицы траекторий с описанием действий
print("\n" + "="*100)
print("ДЕТАЛЬНЫЕ ТРАЕКТОРИИ СО ЗНАЧЕНИЕМ КАЖДОГО ДЕЙСТВИЯ")
print("="*100)

for idx, trajectory in enumerate(trajectories, 1):
    student = trajectory['student_info']
    print(f"\nСТУДЕНТ {idx} (ID: {student.get('user_id_encoded')}) - Модуль {student.get('module', 'Unknown')}")
    print("-" * 100)
    
    steps_with_actions = []
    for step in trajectory['steps']:
        action = step.get('action', -1)
        action_desc = describe_action(action, items_meta)
        steps_with_actions.append({
            'Шаг': step.get('step', 0),
            'Действие': action,
            'Описание': action_desc,
            'Награда': f"{step.get('reward', 0):.4f}",
            'Кум.награда': f"{step.get('cumulative_reward', 0):.4f}",
            'Mastery': f"{step.get('mastery', 0):.3f}",
            'Engagement': f"{step.get('engagement', 0):.3f}",
            'SelfReg': f"{step.get('selfregulation', 0):.3f}",
        })
    
    if steps_with_actions:
        steps_df_display = pd.DataFrame(steps_with_actions)
        print(steps_df_display.to_string(index=False))
    print()


ДЕТАЛЬНЫЕ ТРАЕКТОРИИ СО ЗНАЧЕНИЕМ КАЖДОГО ДЕЙСТВИЯ

СТУДЕНТ 1 (ID: 2841) - Модуль BBB
----------------------------------------------------------------------------------------------------
 Шаг  Действие                      Описание Награда Кум.награда Mastery Engagement SelfReg
   1         0   forumng (week vle__forumng)  0.2370      0.2370   0.236      0.087   0.504
   2         2   subpage (week vle__subpage)  0.0960      0.3330   0.305      0.101   0.650
   3         2   subpage (week vle__subpage)  0.0622      0.3952   0.346      0.110   0.738
   4         8               Assessment: TMA  0.1885      0.5838   0.608      0.175   0.603
   5         5 resource (week vle__resource)  0.0025      0.5863   0.528      0.130   0.709
   6         5 resource (week vle__resource)  0.0065      0.5927   0.480      0.103   0.773
   7         5 resource (week vle__resource)  0.0088      0.6016   0.451      0.087   0.812
   8         2   subpage (week vle__subpage)  0.0142      0.6157   0.433    